# 01 — Data discovery

## Objective
Discover all raw CSV files dynamically, parse city and measurement point identifiers from filenames, and record header coordinates, file sizes, row/column counts, schema signature, and temporal coverage. This is an inventory step only; raw source files are never modified.

The original files are in `datasets/` in this workspace. A clean checkout can put them in `data/raw/`; configuration prefers `data/raw/` when populated and otherwise uses `datasets/`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name.lower()=="notebooks" else Path.cwd()
sys.path.insert(0,str(ROOT/"src"))
from climate_pipeline.config import RAW_DIR, PROJECT_DIR, FINAL_DAILY_COLUMNS
from climate_pipeline import run_pipeline, validate_city_monthly, validate_daily_schema, validate_daily_schema
pd.set_option("display.max_columns",60)


## Discover and inventory

The base filename is interpreted as point 1; numeric suffixes support `_N` and space plus `N`. Source spelling variants are normalized through an explicit mapping. Each file's parameter-year table is inspected after the source header marker.

In [2]:
from climate_pipeline.ingest import discover_files
from climate_pipeline.profiling import profile_files
inventory = discover_files(RAW_DIR)
city_summary = profile_files(inventory)
print(f"{len(inventory)} files, {inventory.city.nunique()} cities, {inventory.measurement_point_id.nunique()} point files")
display(city_summary)
display(inventory.head(10))
(PROJECT_DIR/"reports").mkdir(parents=True,exist_ok=True)
inventory.to_csv(PROJECT_DIR/"reports"/"raw_file_inventory.csv",index=False,date_format="%Y-%m-%d")
city_summary.to_csv(PROJECT_DIR/"reports"/"city_measurement_point_summary.csv",index=False,date_format="%Y-%m-%d")


89 files, 8 cities, 20 point files


,city,files,measurement_points,rows,minimum_date,maximum_date,unique_schemas
0,Ahmedabad,9,9,3510,2000-01-01,2025-12-31,1
1,Bengaluru,10,10,3900,2000-01-01,2025-12-31,1
2,Bhopal,10,10,3900,2000-01-01,2025-12-31,1
3,Bhubaneswar,10,10,3900,2000-01-01,2025-12-31,1
4,Chandigarh,10,10,3900,2000-01-01,2025-12-31,1
5,Chennai,10,10,3900,2000-01-01,2025-12-31,1
6,Coimbatore,10,10,3900,2000-01-01,2025-12-31,1
7,Delhi,20,20,7800,2000-01-01,2025-12-31,1


,source_file,city_key,city,measurement_point_id,latitude,longitude,number_of_rows,number_of_columns,date_min,date_max,schema_signature,file_bytes,source
0,ahemdabad_1.csv,ahmedabad,Ahmedabad,1,22.3061,73.1826,390,15,2000-01-01,2025-12-31,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,36901,NASA/POWER
1,ahemdabad_2.csv,ahmedabad,Ahmedabad,2,23.0680,72.5373,390,15,2000-01-01,2025-12-31,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,36921,NASA/POWER
2,ahemdabad_3.csv,ahmedabad,Ahmedabad,3,23.0685,72.5706,390,15,2000-01-01,2025-12-31,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,36922,NASA/POWER
3,ahemdabad_4.csv,ahmedabad,Ahmedabad,4,23.0464,72.5673,390,15,2000-01-01,2025-12-31,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,36922,NASA/POWER
4,ahemdabad_5.csv,ahmedabad,Ahmedabad,5,23.0027,72.5904,390,15,2000-01-01,2025-12-31,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,36922,NASA/POWER
5,ahemdabad_6.csv,ahmedabad,Ahmedabad,6,22.9882,72.6179,390,15,2000-01-01,2025-12-31,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,36959,NASA/POWER
6,ahemdabad_7.csv,ahmedabad,Ahmedabad,7,23.0438,72.6251,390,15,2000-01-01,2025-12-31,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,36921,NASA/POWER
7,ahemdabad_8.csv,ahmedabad,Ahmedabad,8,23.1041,72.5722,390,15,2000-01-01,2025-12-31,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,36916,NASA/POWER
8,ahemdabad_9.csv,ahmedabad,Ahmedabad,9,22.9632,72.5725,390,15,2000-01-01,2025-12-31,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,36960,NASA/POWER
9,bengaluru 2.csv,bengaluru,Bengaluru,2,13.0233,77.5411,390,15,2000-01-01,2025-12-31,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,36894,NASA/POWER


## Discovery interpretation

The date ranges in the inventory describe calendar years represented by monthly/annual summary tables, not daily readings. See notebook 02 for schema and value profiling.